In [1]:
import re

from pan26_genai_detection.baselines.tfidf import TfidfDetector

In [2]:
text = '''Is TV Too White?

The television industry has long been criticized for its lack of diversity and representation of minority groups. The phrase "TV is too white" has become a rallying cry for those who argue that the medium does not accurately reflect the multicultural society we live in. But is this criticism fair? And what can be done to address the issue of underrepresentation on our screens?

On the surface, it appears that TV is indeed too white. A quick glance at the cast of popular shows reveals a sea of Caucasian faces, with minority actors often relegated to supporting roles or stereotypes. This lack of diversity is not limited to the actors themselves, but also extends to the writers' rooms and production teams. The result is a narrow and homogenous perspective that fails to capture the complexities and richness of diverse experiences.

The consequences of this lack of representation are far-reaching. For one, it can lead to a lack of role models and inspiration for young people of color, who may feel that they do not see themselves reflected on screen. This can have a profound impact on self-esteem and a sense of belonging. Furthermore, the absence of diverse voices and perspectives can result in a lack of nuanced and accurate portrayals of minority communities, perpetuating stereotypes and reinforcing harmful biases.

However, there are signs that the tide is turning. In recent years, there has been a surge in diverse programming, with shows such as "Black-ish", "Master of None", and "Sense8" showcasing complex and multidimensional characters of color. These shows have not only received critical acclaim but also attracted large and devoted audiences, demonstrating that there is a hunger for diverse stories and characters.

Moreover, there are efforts underway to increase diversity behind the camera as well. Initiatives such as the Writers' Guild of America's diversity program and the Directors' Guild of America's diversity and inclusion initiative aim to increase the number of minority writers, directors, and producers in the industry. These programs provide training, mentorship, and job opportunities for underrepresented groups, helping to create a more inclusive and diverse industry.

Despite these positive developments, there is still much work to be done. The television industry must continue to prioritize diversity and inclusion, not just as a moral imperative but also as a business necessity. In a rapidly changing media landscape, where audiences are increasingly diverse and discerning, TV shows that fail to reflect this diversity risk losing relevance and viewers.

To address the issue of underrepresentation on TV, the industry must adopt a multifaceted approach. This includes increasing diversity in casting, writing rooms, and production teams, as well as providing opportunities for minority talent to develop and showcase their work. Additionally, TV shows must strive to portray diverse characters and communities in a nuanced and accurate way, avoiding stereotypes and tropes.

In conclusion, the criticism that "TV is too white" is a valid one. The industry has a long way to go in terms of representation and diversity. However, with the efforts underway to increase diversity and inclusion, there is hope for a more vibrant and representative TV landscape in the future. By prioritizing diversity and providing opportunities for underrepresented groups, the television industry can create a more accurate and engaging reflection of our multicultural society, ultimately enriching the viewing experience for all.
'''

# text = ' '.join(map(lambda x: x[::-1], text.split(' ')))
# text = re.sub(r'(\s+)([.,?!])', r'\2\1', text)
# print(text)

det = TfidfDetector()
det.predict_with_score(text)

/home/roce3528/.cache/pypoetry/virtualenvs/pan26-generative-authorship-detection-VB44w5UW-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/roce3528/.cache/pypoetry/virtualenvs/pan26-generative-authorship-detection-VB44w5UW-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/roce3528/.cache/pypoetry/virtualenvs/pa

(np.int32(1), np.float32(0.60456324))

In [3]:
import numpy as np

features = det.vec.inverse_transform(det.vec.transform([text]))[0]
feature_idx = set([det.vec.vocabulary_[k] for k in features])
feature_idx = [i for i in det.clf.coef_.argsort()[::-1][0] if i in feature_idx]
features_sorted = det.vec.get_feature_names_out()[[int(i) for i in det.clf.coef_.argsort()[::-1][0] if i in feature_idx]]

In [4]:
import re

substr = '|'.join(features_sorted[:100])
print(re.sub(rf'\b({substr})\b', 'x', text, re.IGNORECASE))
# print(rf'\b({substr})\b')

Is TV Too White?

The television industry x x been criticized for its lack of diversity and representation of minority groups. The phrase "TV is too white" has become a rallying cry for those who argue that the medium does not accurately reflect the multicultural society we live in. But is this criticism fair? And what can be done to address the issue of underrepresentation on our screens?

On the surface, it appears that TV is indeed too white. A quick glance at the cast of popular shows reveals a sea of Caucasian faces, with minority actors often relegated to supporting roles or stereotypes. This lack of diversity is not limited to the actors themselves, but also extends to the writers' rooms and production teams. The result is a narrow and homogenous perspective that fails to capture the complexities and richness of diverse experiences.

The consequences of this lack of representation are far-reaching. For one, it can lead to a lack of role models and inspiration for young people of

In [5]:
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils import shuffle

from pathlib import Path

data_path = Path('../data/text-llm')
data_path_not_wm = data_path / 'llama-3.3-70b-instruct-8bit/nytimes-writing-prompts'
data_path_wm = data_path / 'llama-3.3-70b-instruct-8bit-watermark-gamma25-bias2-temp1/nytimes-writing-prompts'
# data_path_wm = data_path / 'llama-3.3-70b-instruct-8bit-watermark-gamma05-bias5-temp1/nytimes-writing-prompts'

files_not_wm = sorted([p for p in data_path_not_wm.glob('*.txt') if data_path_wm.joinpath(p.name).exists()])
files_wm = sorted([p for p in data_path_wm.glob('*.txt') if data_path_not_wm.joinpath(p.name).exists()])

assert len(files_wm) == len(files_not_wm)
for w, n in zip(files_wm, files_not_wm):
    assert w.name == n.name

data_wm = [f.read_text() for f in files_wm]
data_not_wm = [f.read_text() for f in files_not_wm]

X = [*data_not_wm, *data_wm]
y = [0] * len(data_not_wm) + [1] * len(data_wm)

ngram_range = (1, 4)
analyzer = 'word'
max_features = 500

def with_train_test_split():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram_range, analyzer=analyzer, max_features=max_features)),
        ('clf', LinearSVC())
    ])
    pipe.fit(X_train, y_train)

    print('Train / test:')
    print('Acc', round(accuracy_score(y_test, pipe.predict(X_test)), 3))
    print('F1', round(f1_score(y_test, pipe.predict(X_test)), 3))

def with_cross_validation():
    _X, _y = shuffle(X, y, random_state=42)

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram_range, analyzer=analyzer, max_features=max_features)),
        ('clf', LinearSVC())
    ])

    scoring = {
        'accuracy': make_scorer(accuracy_score),
        'f1': make_scorer(f1_score),
    }
    scores = cross_validate(pipe, _X, _y, cv=5, scoring=scoring)
    print('Cross-validation')
    print('Acc', round(scores['test_accuracy'].mean(), 3))
    print('F1', round(scores['test_f1'].mean(), 3))

with_train_test_split()
print()
with_cross_validation()

Train / test:
Acc 0.4
F1 0.4

Cross-validation
Acc 0.43
F1 0.419
